# Q2A | Summarize the Data

> *Summarize the data. How much data is present? What attributes/features are continuous valued? Which attributes are categorical?*

This notebook answers the question in three parts:

1. **How much data is present?** Row and column counts, duplicates, and missing values.
2. **Which attributes are continuous?** Data types, the number of distinct values, and the range of each numeric attribute.
3. **Which attributes are categorical?** A simple classification rule applied to every column, a manual review of its result, and the final classification.

The **Answer Summary** below pulls the three parts together.

---

#### **IMPORTANT**

This dataset is a single file with no `train` / `test` split. The split is made in a later notebook, so this summary describes the whole dataset. It only counts and describes the data. No modeling decisions are made here, so looking at rows that will later become test data does not introduce `data snooping` bias.

# Answer Summary

- **How much data is present?** 731 rows and 16 columns (1 target, `Rentals`, and 15 other attributes) in a single file with no train/test split. Each row is one day, from 2011-01-01 to 2012-12-31. There are no duplicate rows and no missing values. Two of the attributes, `Casual` and `Registered`, add up exactly to the target.
- **Which attributes are continuous?** 7: the four measurements `Temp`, `FeelsLike`, `Humidity`, and `Wind` (decimals between 0 and 1), and the three rental counts `Casual`, `Registered`, and `Rentals` (whole numbers with hundreds of distinct values).
- **Which attributes are categorical?** 7: `Season`, `Year`, `Month`, `IsHoliday`, `Weekday`, `IsWorkday`, and `Weather`. All are stored as whole-number codes with 2 to 12 levels. The remaining 2 columns fit neither group: `ID` is a row identifier and `Date` is a date.

---

### Imports

In [1]:
import pandas as pd

from OLD.Q2.consts import *

from OLD.Q2.utils import read_csv_to_df

### Fetch Data

In [2]:
df = read_csv_to_df(path=DATA_PATH, parse_dates=["dteday"])
df = df.rename(columns=SHORT_NAMES)

# 1. How Much Data is Present?

In [3]:
summary = pd.DataFrame(
	{
		"row count": [ df.shape[0] ],
		"columns count": [ df.shape[1] ],
		"missing values": [ df.isnull().sum().sum() ],
		"rows w/ missing values": [ df.isnull().any(axis=1).sum() ],
		"% rows w/ missing values": [
			round(df.isnull().any(axis=1).sum() / df.shape[0] * 100, 2),
		]
	},
	index=["Full Dataset"],
)
s = summary.T
s

,Full Dataset
row count,731.0
columns count,16.0
missing values,0.0
rows w/ missing values,0.0
% rows w/ missing values,0.0


In [4]:
attributes = [col for col in df.columns if col != TARGET]

print(
	f"There are {s.at['row count', 'Full Dataset']:,.0f} rows and {s.at['columns count', 'Full Dataset']:.0f} columns."
)

print(
	f"Of the {s.at['columns count', 'Full Dataset']:.0f} columns, 1 is the target ({TARGET}) and {len(attributes)} are other attributes."
)

print(
	f"There are {s.at['missing values', 'Full Dataset']:,.0f} missing values affecting {s.at['rows w/ missing values', 'Full Dataset']:,.0f} rows ({s.at['% rows w/ missing values', 'Full Dataset']:.2f}%)."
)

print(
	f"The rows cover {df['Date'].min():%Y-%m-%d} to {df['Date'].max():%Y-%m-%d}, with {df['Date'].nunique()} distinct dates."
)

print(
	f"Casual + Registered equals {TARGET} in every row: {(df['Casual'] + df['Registered'] == df[TARGET]).all()}"
)

There are 731 rows and 16 columns.
Of the 16 columns, 1 is the target (Rentals) and 15 are other attributes.
There are 0 missing values affecting 0 rows (0.00%).
The rows cover 2011-01-01 to 2012-12-31, with 731 distinct dates.
Casual + Registered equals Rentals in every row: True


In [5]:
df.head(3).T

,0,1,2
ID,1,2,3
Date,2011-01-01 00:00:00,2011-01-02 00:00:00,2011-01-03 00:00:00
Season,1,1,1
Year,0,0,0
Month,1,1,1
IsHoliday,0,0,0
Weekday,6,0,1
IsWorkday,0,0,1
Weather,2,2,1
Temp,0.344167,0.363478,0.196364


**What a row contains**

- The column headers are the short names set in `Q2/consts.py` (ex: the file's `cnt` column is `Rentals` here).
- Each row is one day. There are 731 rows and 731 distinct dates, from 2011-01-01 to 2012-12-31.
- The target, `Rentals`, is the total number of bike rentals on that day.
- `Casual` and `Registered` add up to `Rentals` in every row. They are the two parts of the target, so using them as input features would give the answer away.
- `ID` is a row counter and `Date` is the date.
- The remaining columns describe the day: its calendar (`Season`, `Year`, `Month`, `IsHoliday`, `Weekday`, `IsWorkday`) and its weather (`Weather`, `Temp`, `FeelsLike`, `Humidity`, `Wind`).
- Which columns are used as input features is decided in later notebooks.

**No train/test split**

- The data comes as one file, so there is no train or test column in the table above. The split is made in a later notebook.

**Missing values**

- There are **no missing values**, so no rows need to be dropped or filled.

## Duplicate Rows

In [13]:
df.duplicated().sum(),

(0,)

There are **no duplicate rows**. Every row is a distinct day.

## Missing Values by Feature

In [7]:
missing_by_feature = pd.DataFrame({
	'Missing': df.isna().sum(),
	'% of rows': (df.isna().sum() / len(df) * 100).round(2),
}).sort_values('Missing', ascending=False)

print(f"{(missing_by_feature['Missing'] > 0).sum()} of {len(missing_by_feature)} columns have missing values.")
missing_by_feature

0 of 16 columns have missing values.


,Missing,% of rows
ID,0,0.0
Date,0,0.0
Season,0,0.0
Year,0,0.0
Month,0,0.0
IsHoliday,0,0.0
Weekday,0,0.0
IsWorkday,0,0.0
Weather,0,0.0
Temp,0,0.0


- **0 of 16** columns have missing values. Every column, including the target, is complete in all 731 rows.
- Because nothing is missing, there is no breakdown of missing values per row to look at.

# 2. Which Attributes are Continuous?

Three checks are used to call an attribute continuous:

1. **Data type** rules out text and dates. A column holding strings would load as `object`, and the date column is parsed as `datetime64[ns]`. Only numeric columns can be continuous.
2. **Number of distinct values** rules out numeric codes. A category stored as numbers (ex: 1, 2, 3) still loads as a number, so a numeric data type alone does not prove a column is continuous.
3. **Whole numbers only** (in the Value Ranges table) separates codes and counts from measured values.

In [8]:
info_df = pd.DataFrame({
	"dtype": df.dtypes,
	"non-null values": df.count(),
	"unique values": df.nunique(),
	"% unique (of non-null)": (df.nunique() / df.count() * 100).round(2),
	"most frequent value count": df.apply(lambda col: col.value_counts().iloc[0]),
})

print(df.dtypes.astype(str).value_counts().to_string())
info_df

int64             11
float64            4
datetime64[ns]     1


,dtype,non-null values,unique values,% unique (of non-null),most frequent value count
ID,int64,731,731,100.00,1
Date,datetime64[ns],731,731,100.00,1
Season,int64,731,4,0.55,188
Year,int64,731,2,0.27,366
Month,int64,731,12,1.64,62
IsHoliday,int64,731,2,0.27,710
Weekday,int64,731,7,0.96,105
IsWorkday,int64,731,2,0.27,500
Weather,int64,731,3,0.41,463
Temp,float64,731,499,68.26,5


- **15 of the 16 columns are numeric** (11 `int64` and 4 `float64`), and 1 is a date. There are no text columns, so the data type alone separates almost nothing here.
- **The number of distinct values splits the numeric columns into two clear groups:**
  - 7 columns have 12 or fewer distinct values: `Season` (4), `Year` (2), `Month` (12), `IsHoliday` (2), `Weekday` (7), `IsWorkday` (2), and `Weather` (3).
  - 8 columns have 499 or more: `ID` (731), `Temp` (499), `FeelsLike` (690), `Humidity` (595), `Wind` (650), `Casual` (606), `Registered` (679), and `Rentals` (696).
  - No column falls in between.
- In the second group no single value dominates. The most common value in any of those columns repeats at most 5 times in 731 rows.
- `ID` and `Date` are 100.00% unique: every row has its own value.
- `IsHoliday` is very uneven. Its most common value appears in 710 of the 731 rows.

## Value Ranges

In [9]:
numeric_df = df.select_dtypes(include="number")

feature_dictionary = pd.DataFrame({
	"role": ["target" if col == TARGET else "attribute" for col in numeric_df.columns],
	"min": numeric_df.min().round(3),
	"max": numeric_df.max().round(3),
	"mean": numeric_df.mean().round(3),
	"std": numeric_df.std().round(3),
	"whole numbers only": numeric_df.apply(lambda col: (col.dropna() % 1 == 0).all()),
}, index=numeric_df.columns)

feature_dictionary

,role,min,max,mean,std,whole numbers only
ID,attribute,1.000,731.000,366.000,211.166,True
Season,attribute,1.000,4.000,2.497,1.111,True
Year,attribute,0.000,1.000,0.501,0.500,True
Month,attribute,1.000,12.000,6.520,3.452,True
IsHoliday,attribute,0.000,1.000,0.029,0.167,True
Weekday,attribute,0.000,6.000,2.997,2.005,True
IsWorkday,attribute,0.000,1.000,0.684,0.465,True
Weather,attribute,1.000,3.000,1.395,0.545,True
Temp,attribute,0.059,0.862,0.495,0.183,False
FeelsLike,attribute,0.079,0.841,0.474,0.163,False


Only the 15 numeric columns are shown. The date column is left out.

- **4 columns hold decimals:** `Temp`, `FeelsLike`, `Humidity`, and `Wind`. These are the four `float64` columns. The other 11 hold whole numbers only.
- **Measurements:** all four decimal columns sit between 0 and 1. `Temp` runs from 0.059 to 0.862, `FeelsLike` from 0.079 to 0.841, `Humidity` from 0.0 to 0.972, and `Wind` from 0.022 to 0.507. They look already scaled.
- **Counts:** `Casual` (2 to 3,410), `Registered` (20 to 6,946), and the target `Rentals` (22 to 8,714) are whole numbers spread over a wide range.
- **Codes:** `Season` (1 to 4), `Year` (0 to 1), `Month` (1 to 12), `IsHoliday` (0 to 1), `Weekday` (0 to 6), `IsWorkday` (0 to 1), and `Weather` (1 to 3) are whole numbers in a short fixed range. A mean or standard deviation of a code (ex: 2.497 for `Season`) is not meaningful.
- `ID` runs from 1 to 731, the same as the number of rows.
- `Humidity` has a minimum of exactly 0.0, which is worth a closer look in the next notebook.
- The columns use very different ranges (0 to 1 vs. 22 to 8,714), which is worth noting for any model that is sensitive to feature scale.

**The 4 measurements and the 3 rental counts are the continuous valued attributes.** The counts are whole numbers, but they take hundreds of distinct values across a wide range, so they are grouped with the continuous attributes and not with the categorical ones.

# 3. Which Attributes are Categorical?

To make the classification explicit, each column is first tested with a simple rule:

> A column is **categorical** if its data type is text (`string` / `object` / `category`), or if it has 20 or fewer distinct values. Otherwise it is **continuous**.

The threshold of 20 is generous. Every column here has either 12 or fewer distinct values or 499 or more, so the result does not depend on the exact cutoff. The rule has no label for an identifier or a date, so its result is reviewed below.

In [10]:
CATEGORICAL_MAX_UNIQUE = 20

classification = pd.DataFrame({
	"dtype": df.dtypes,
	"unique values": df.nunique(),
})
classification["text dtype"] = [
	pd.api.types.is_string_dtype(dtype) or isinstance(dtype, pd.CategoricalDtype)
	for dtype in classification["dtype"]
]
classification["<= 20 unique"] = classification["unique values"] <= CATEGORICAL_MAX_UNIQUE
classification["classification"] = [
	"categorical" if is_text or is_few else "continuous"
	for is_text, is_few in zip(classification["text dtype"], classification["<= 20 unique"])
]

print(classification["classification"].value_counts().to_string())
classification

classification
continuous     9
categorical    7


,dtype,unique values,text dtype,<= 20 unique,classification
ID,int64,731,False,False,continuous
Date,datetime64[ns],731,False,False,continuous
Season,int64,4,False,True,categorical
Year,int64,2,False,True,categorical
Month,int64,12,False,True,categorical
IsHoliday,int64,2,False,True,categorical
Weekday,int64,7,False,True,categorical
IsWorkday,int64,2,False,True,categorical
Weather,int64,3,False,True,categorical
Temp,float64,499,False,False,continuous


The rule labels **9 columns continuous** and **7 categorical**.

- No column is text. All 7 categorical columns are caught by the second check: they are numbers with 20 or fewer distinct values.
- Two of the continuous labels need a second look:
  - **`ID`:** the rule says continuous because it is numeric with 731 distinct values in 731 rows. A value that never repeats is a row identifier and not a measurement.
  - **`Date`:** the rule says continuous because it is not text and has 731 distinct values. It is a date (`datetime64[ns]`) and not a number, so it is kept as its own group.

## Category Levels

In [11]:
few_unique_cols = classification.index[classification["<= 20 unique"]]

for col in few_unique_cols:
	print(f"{col}: {df[col].value_counts().sort_index().to_dict()}")

Season: {1: 181, 2: 184, 3: 188, 4: 178}
Year: {0: 365, 1: 366}
Month: {1: 62, 2: 57, 3: 62, 4: 60, 5: 62, 6: 60, 7: 62, 8: 62, 9: 60, 10: 62, 11: 60, 12: 62}
IsHoliday: {0: 710, 1: 21}
Weekday: {0: 105, 1: 105, 2: 104, 3: 104, 4: 104, 5: 104, 6: 105}
IsWorkday: {0: 231, 1: 500}
Weather: {1: 463, 2: 247, 3: 21}


These are the 7 columns with 20 or fewer distinct values. Each is stored as a whole-number code, so the numbers are labels and not quantities.

- **Two levels:** `Year` (365 and 366 rows), `IsHoliday` (710 and 21 rows), and `IsWorkday` (231 and 500 rows).
- **`Season`** has 4 levels with 178 to 188 rows each.
- **`Weekday`** has 7 levels with 104 or 105 rows each.
- **`Month`** has 12 levels with 57 to 62 rows each.
- **`Weather`** has 3 levels and is uneven: 463, 247, and 21 rows.
- `IsHoliday` and `Weather` each have one level with only 21 rows.

## Final Classification

In [12]:
MANUAL_REVIEW = {
	"ID": "identifier",
	"Date": "date",
}

final = classification[["dtype", "unique values"]].copy()
final["rule result"] = classification["classification"]
final["final type"] = pd.Series(MANUAL_REVIEW).reindex(final.index).fillna(final["rule result"])

for kind, cols in final.groupby("final type").groups.items():
	print(f"{kind} ({len(cols)}): {', '.join(cols)}")

final[final["rule result"] != final["final type"]]

categorical (7): Season, Year, Month, IsHoliday, Weekday, IsWorkday, Weather
continuous (7): Temp, FeelsLike, Humidity, Wind, Casual, Registered, Rentals
date (1): Date
identifier (1): ID


,dtype,unique values,rule result,final type
ID,int64,731,continuous,identifier
Date,datetime64[ns],731,continuous,date


After the review, the 16 columns split into four groups:

- **Continuous (7):**
  - Measurements: `Temp`, `FeelsLike`, `Humidity`, `Wind`.
  - Counts: `Casual`, `Registered`, `Rentals`.
- **Categorical (7):** `Season`, `Year`, `Month`, `IsHoliday`, `Weekday`, `IsWorkday`, `Weather`.
- **Identifier (1):** `ID`.
- **Date (1):** `Date`.

**Notes**

- All 7 categorical columns are stored as whole numbers. A model would read them as quantities unless they are handled as categories in a later notebook.
- Of the 7 continuous columns, `Rentals` is the target, and `Casual` and `Registered` add up to it (shown in part 1).
- `Date` is not a number as it is, but `Year`, `Month`, and `Weekday` already describe the date.